# Assignment # 01 
# SP24-BDS-044 Rahemeen Ilyas
# Natural Language Processing (Lab)




## Setup — install packages & generate sample files

In [1]:
import os, re

In [45]:
PDF_FILE   = "sample_document.pdf"     # used in Tasks 1-3 
DOCX_A     = "sample_doc_A.docx"       # used in Tasks 4, 5 & 6 
DOCX_B     = "sample_doc_B.docx"       # used in Task 6

USERS_URL    = "https://jsonplaceholder.typicode.com/users"
POSTS_URL    = "https://jsonplaceholder.typicode.com/posts"
TODOS_URL    = "https://jsonplaceholder.typicode.com/todos"
COMMENTS_URL = "https://jsonplaceholder.typicode.com/comments"
print("Paths configured")

Paths configured


## Task 1: Multi-Page PDF Extraction with Word Frequency Analysis

#### (a) Open the PDF and extract text page by page

In [46]:
from pypdf import PdfReader

reader = PdfReader(PDF_FILE)
pages_list = []                       # list of strings; index i = page i+1
for page in reader.pages:
    pages_list.append(page.extract_text() or "")

print(f"Total pages extracted: {len(pages_list)}")
print(f"Page 1 preview: {pages_list[0][:100]}...")

Total pages extracted: 6
Page 1 preview: The quick brown fox jumps over the lazy dog. Natural language processing is a fascinating field of a...


#### (b) Merge pages, clean with `re.sub()`, and tokenize

In [47]:
full_text = " ".join(pages_list)

# remove punctuation and digits, lowercase, split on whitespace
cleaned = re.sub(r"[^\w\s]", " ", full_text)   # punctuation -> space
cleaned = re.sub(r"\d+", " ", cleaned)          # digits -> space
cleaned = cleaned.lower()

tokens = cleaned.split()
print(f"Total token count: {len(tokens)}")
print("First 15 tokens:", tokens[:15])

Total token count: 303
First 15 tokens: ['the', 'quick', 'brown', 'fox', 'jumps', 'over', 'the', 'lazy', 'dog', 'natural', 'language', 'processing', 'is', 'a', 'fascinating']


#### (c) Word frequency dictionary, stopword removal, top 10

In [49]:
# Build frequency dictionary 
freq = {}
for t in tokens:
    freq[t] = freq.get(t, 0) + 1

# Remove 28 common stopwords
STOPWORDS = {
    "the", "is", "a", "an", "and", "or", "of", "to", "in", "on",
    "for", "with", "are", "was", "were", "be", "been", "it", "this", "that",
    "as", "at", "by", "from", "can", "will", "has", "have", "had",
}
for sw in STOPWORDS:
    freq.pop(sw, None)

top10 = sorted(freq.items(), key=lambda kv: kv[1], reverse=True)[:10]
print("Top 10 most frequent words:")
for word, count in top10:
    print(f"  {word:<15} {count}")

Top 10 most frequent words:
  page            6
  learning        4
  nlp             4
  models          3
  data            3
  natural         2
  language        2
  processing      2
  field           2
  machine         2


#### (d) Keyword search across pages (with page numbers)

In [50]:
keyword = input("Enter a keyword to search: ").strip().lower()

found = False
for page_no, page_text in enumerate(pages_list, start=1):
    sentences = re.split(r"(?<=[.!?])\s+", page_text)
    for sent in sentences:
        if keyword in sent.lower():
            print(f"[Page {page_no}] {sent.strip()}")
            found = True

if not found:
    print(f"No sentences containing '{keyword}' were found in the document.")

Enter a keyword to search:  Natural


[Page 1] Natural language processing is a fascinating field of artificial intelligence.
[Page 3] Deep learning has revolutionized natural language processing.


## Task 2: Structured Page Metadata Extraction and CSV Export

#### (a) Build `pages_data` list of dictionaries

In [52]:
pages_data = []
for page_no, page_text in enumerate(pages_list, start=1):
    words = page_text.split()
    pages_data.append({
        "page_number": page_no,
        "raw_text": page_text,
        "word_count": len(words),
        "char_count": len(page_text.replace(" ", "")),  # excluding spaces
    })

print("First 2 entries:")
for entry in pages_data[:2]:
    print(entry)


First 2 entries:
{'page_number': 1, 'raw_text': 'The quick brown fox jumps over the lazy dog. Natural language processing is a fascinating field of artificial intelligence. It combines linguistics and machine learning. Many companies build chatbots using NLP techniques. Visit https://example.com for more details. The system achieved 95% accuracy on the test set in 2023.\n- Page 1 -\n', 'word_count': 52, 'char_count': 285}
{'page_number': 2, 'raw_text': 'Machine learning models require large amounts of training data. The data must be cleaned and prepared before training. Preprocessing is a critical step in any NLP pipeline. Stopwords are commonly removed to reduce noise. Tokenization splits text into smaller units called tokens. These tokens form the vocabulary of the model.\n- Page 2 -\n', 'word_count': 54, 'char_count': 286}


#### (b) Highest / lowest non-zero / average word count

In [53]:
# (i) page with the HIGHEST word count + its full content
highest = max(pages_data, key=lambda p: p["word_count"])
print(f"(i) Highest word count: Page {highest['page_number']} ({highest['word_count']} words)")
print(f"    Content: {highest['raw_text']}\n")

# (ii) page with the lowest NON-ZERO word count (zero-word pages are skipped safely)
non_zero = [p for p in pages_data if p["word_count"] > 0]
if non_zero:
    lowest = min(non_zero, key=lambda p: p["word_count"])
    print(f"(ii) Lowest non-zero word count: Page {lowest['page_number']} ({lowest['word_count']} words)")
else:
    print("(ii) All pages have zero words.")

# (iii) average word count rounded to 2 dp
avg = round(sum(p["word_count"] for p in pages_data) / len(pages_data), 2)
print(f"(iii) Average word count across all pages: {avg}")

(i) Highest word count: Page 5 (60 words)
    Content: Sentiment analysis determines the emotional tone of a text. It is widely used in product reviews and social media monitoring. The classifier assigns positive, negative, or neutral labels. Customer feedback helps companies improve their products. A rating of 4.5 stars usually indicates a positive sentiment. However, sarcasm remains a difficult challenge for automated systems in 2024.
- Page 5 -


(ii) Lowest non-zero word count: Page 3 (49 words)
(iii) Average word count across all pages: 54.0


#### (c) Phrase search with custom `PhraseNotFoundError`

In [54]:
class PhraseNotFoundError(Exception):
    pass   # raised when a phrase is not found on any page

phrase = input("Enter a search phrase: ").strip()

try:
    found = False
    for p in pages_data:
        count = p["raw_text"].lower().count(phrase.lower())
        if count > 0:
            print(f"Page {p['page_number']}: '{phrase}' appears {count} time(s)")
            found = True
    if not found:
        raise PhraseNotFoundError(f"Phrase '{phrase}' not found on any page.")
except PhraseNotFoundError as e:
    print(f"PhraseNotFoundError caught gracefully: {e}")

Enter a search phrase:  NLP


Page 1: 'NLP' appears 1 time(s)
Page 2: 'NLP' appears 1 time(s)
Page 4: 'NLP' appears 1 time(s)
Page 6: 'NLP' appears 1 time(s)


#### (d) Write and re-read `pdf_summary.csv` (csv module only)

In [55]:
import csv

with open("pdf_summary.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["page_number", "word_count", "char_count"])
    writer.writeheader()
    for p in pages_data:
        writer.writerow({k: p[k] for k in ["page_number", "word_count", "char_count"]})

# Re-read to confirm the write succeeded
print("Contents of pdf_summary.csv:")
with open("pdf_summary.csv", "r", encoding="utf-8") as f:
    reader = csv.reader(f)
    for row in reader:
        print(row)

Contents of pdf_summary.csv:
['page_number', 'word_count', 'char_count']
['1', '52', '285']
['2', '54', '286']
['3', '49', '294']
['4', '53', '298']
['5', '60', '339']
['6', '56', '323']


## Task 3: PDF Text Preprocessing Pipeline for NLP

#### (a) Cleaning with `re.sub()` — URLs, digits, non-alpha, spaces

In [56]:
raw = " ".join(pages_list)
before = len(raw)

text = raw
text = re.sub(r"https?://\S+|www\.\S+", " ", text)   # (i) remove URLs
text = re.sub(r"\d+", " ", text)                       # (ii) remove digits
text = re.sub(r"[^a-zA-Z. ]", " ", text)                # (iii) keep alpha, spaces, full stops
text = re.sub(r"\s+", " ", text).strip()               # (iv) collapse multiple spaces

after = len(text)
print(f"Character count BEFORE cleaning: {before}")
print(f"Character count AFTER  cleaning: {after}")

Character count BEFORE cleaning: 2142
Character count AFTER  cleaning: 2053


#### (b) Lowercase + sentence splitting

In [57]:
text = text.lower()

sentences = re.split(r"[.!?]", text)
sentences = [s.strip() for s in sentences]              # strip each sentence
before_count = len(sentences)
sentences = [s for s in sentences if s]                 # remove empty strings
print(f"Sentence count BEFORE removing empties: {before_count}")
print(f"Sentence count AFTER  removing empties: {len(sentences)}")

Sentence count BEFORE removing empties: 39
Sentence count AFTER  removing empties: 39


#### (c) Tokenize, remove stopwords, filter short sentences

In [58]:
import nltk
nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords

EN_STOPWORDS = set(stopwords.words("english"))

filtered_sentences = []
for s in sentences:
    words = s.split()
    kept = [w for w in words if w not in EN_STOPWORDS]
    if len(kept) >= 5:                     # keep only sentences with >= 5 non-stopword tokens
        filtered_sentences.append(" ".join(kept))

print(f"Sentences BEFORE stopword filter: {len(sentences)}")
print(f"Sentences AFTER  stopword filter: {len(filtered_sentences)}")
print()
print("3 sample sentences:")
for s in filtered_sentences[:3]:
    print(" -", s)

Sentences BEFORE stopword filter: 39
Sentences AFTER  stopword filter: 28

3 sample sentences:
 - quick brown fox jumps lazy dog
 - natural language processing fascinating field artificial intelligence
 - many companies build chatbots using nlp techniques


#### (d) Write `nlp_ready_output.txt` and report stats

In [60]:
with open("nlp_ready_output.txt", "w", encoding="utf-8") as f:
    for s in filtered_sentences:
        f.write(s + "\n")

with open("nlp_ready_output.txt", "r", encoding="utf-8") as f:
    lines = f.read().splitlines()

print(f"(i) Total lines written: {len(lines)}")
print(f"(ii) Longest sentence ({len(max(lines, key=len))} chars): {max(lines, key=len)}")
print(f"(iii) Shortest sentence ({len(min(lines, key=len))} chars): {min(lines, key=len)}")

(i) Total lines written: 28
(ii) Longest sentence (69 chars): natural language processing fascinating field artificial intelligence
(iii) Shortest sentence (30 chars): quick brown fox jumps lazy dog


## Task 4: DOCX Paragraph Classification and Statistical Analysis

#### (a) Extract paragraphs with styles

In [61]:
from docx import Document

doc = Document(DOCX_A)
paragraphs = [{"text": p.text, "style": p.style.name} for p in doc.paragraphs]

unique_styles = sorted({p["style"] for p in paragraphs})
print(f"Total paragraph count: {len(paragraphs)}")
print(f"Unique style names: {unique_styles}")

Total paragraph count: 8
Unique style names: ['Heading 1', 'Heading 2', 'Normal']


#### (b) Classify into Headings / Body / Other-Empty

In [62]:
headings, body, other_empty = [], [], []
for p in paragraphs:
    if p["style"].startswith("Heading"):
        headings.append(p)
    elif p["style"] == "Normal" or "Body" in p["style"]:
        body.append(p)
    else:
        other_empty.append(p)

print(f"(i)   Headings           : {len(headings)}")
print(f"(ii)  Body               : {len(body)}")
print(f"(iii) Other/Empty        : {len(other_empty)}")

(i)   Headings           : 3
(ii)  Body               : 5
(iii) Other/Empty        : 0


#### (c) Word/char stats for Body paragraphs with > 5 words

In [63]:
qualifying = []
for p in body:
    wc = len(p["text"].split())
    if wc > 5:
        cc = len(p["text"].replace(" ", ""))   # chars excluding spaces
        qualifying.append({"text": p["text"], "word_count": wc, "char_count": cc})

if qualifying:
    highest_wc = max(qualifying, key=lambda x: x["word_count"])
    lowest_wc  = min(qualifying, key=lambda x: x["word_count"])
    mean_wc    = round(sum(x["word_count"] for x in qualifying) / len(qualifying), 2)

    print(f"(i)   Highest word count ({highest_wc['word_count']} words): "
          f"{highest_wc['text'][:90]}...")
    print(f"(ii)  Lowest word count above 5 ({lowest_wc['word_count']} words): "
          f"{lowest_wc['text'][:90]}...")
    print(f"(iii) Mean word count: {mean_wc}")
else:
    print("No body paragraphs with more than 5 words.")

(i)   Highest word count (29 words): Modern NLP relies heavily on deep learning architectures. Recurrent neural networks were p...
(ii)  Lowest word count above 5 (19 words): The table above summarizes popular NLP applications. Each application presents unique chal...
(iii) Mean word count: 25.25


#### (d) Nested style dictionary, sorted, pretty-printed

In [64]:
import pprint

style_dict = {}
for p in paragraphs:
    style_dict.setdefault(p["style"], {"count": 0, "texts": []})
    style_dict[p["style"]]["count"] += 1
    style_dict[p["style"]]["texts"].append(p["text"])

# Sort outer dict by count descending
sorted_style_dict = dict(sorted(style_dict.items(), key=lambda kv: kv[1]["count"], reverse=True))

# If a style has more than 2 paragraphs, keep only the first 2 texts for display
for style, info in sorted_style_dict.items():
    if info["count"] > 2:
        info["texts"] = info["texts"][:2]

pprint.pprint(sorted_style_dict, width=100)

{'Heading 1': {'count': 1, 'texts': ['Introduction to Natural Language Processing']},
 'Heading 2': {'count': 2, 'texts': ['Core Techniques', 'Common Applications']},
 'Normal': {'count': 5,
            'texts': ['Natural language processing enables computers to understand human language. '
                      'It is a subfield of artificial intelligence and linguistics. Applications '
                      'include machine translation, sentiment analysis, and question answering '
                      'systems.',
                      'Modern NLP relies heavily on deep learning architectures. Recurrent neural '
                      'networks were popular before transformers arrived. Today, transformer-based '
                      'models dominate the field and achieve state of the art results.']}}


## Task 5: DOCX Table Extraction and Pandas Integration

#### (a) Extract every table as a list of lists

In [65]:
import pandas as pd

tables_data = []
for tbl in doc.tables:
    table_rows = []
    for row in tbl.rows:
        table_rows.append([cell.text for cell in row.cells])
    tables_data.append(table_rows)

print(f"Total tables found: {len(tables_data)}")
for i, t in enumerate(tables_data, 1):
    print(f"  Table {i}: {len(t)} rows x {len(t[0])} columns")

Total tables found: 1
  Table 1: 4 rows x 3 columns


#### (b) First table -> DataFrame + empty-cell count per column

In [66]:
first_table = tables_data[0]
header, data = first_table[0], first_table[1:]
df = pd.DataFrame(data, columns=header)

print("Full DataFrame:")
print(df.to_string(index=False))

print()
empty_per_col = df.apply(lambda col: col.isna().sum() + (col.astype(str).str.strip() == "").sum())
print("Empty/None cells per column:")
print(empty_per_col)

Full DataFrame:
  Application               Description Difficulty
  Translation Convert between languages       High
    Sentiment    Detect emotion in text     Medium
Summarization    Shorten long documents       High

Empty/None cells per column:
Application    0
Description    0
Difficulty     0
dtype: int64


#### (c) Paragraphs before the first table (via `doc.element.body`)

In [67]:
from docx.oxml.ns import qn

pre_table_paras = []
for child in doc.element.body.iterchildren():
    if child.tag == qn("w:tbl"):        # stop at the first table
        break
    if child.tag == qn("w:p"):
        # find matching Paragraph object to get its text
        for p in doc.paragraphs:
            if p._element is child:
                pre_table_paras.append(p.text)
                break

print(f"Paragraphs before the first table: {len(pre_table_paras)}")
for t in pre_table_paras:
    print(" -", t[:80])

Paragraphs before the first table: 7
 - Introduction to Natural Language Processing
 - Natural language processing enables computers to understand human language. It i
 - Modern NLP relies heavily on deep learning architectures. Recurrent neural netwo
 - Core Techniques
 - Tokenization breaks text into individual units. These units may be words, subwor
 -    
 - Common Applications


#### (d) Write `docx_extracted.txt` with labeled sections

In [27]:
with open("docx_extracted.txt", "w", encoding="utf-8") as f:
    f.write("--- PRE-TABLE PARAGRAPHS ---\n")
    for t in pre_table_paras:
        f.write(t + "\n")
    f.write("\n--- TABLE DATA ---\n")
    for row in first_table:
        f.write(" | ".join(row) + "\n")

# Confirm by printing the file back
with open("docx_extracted.txt", "r", encoding="utf-8") as f:
    print(f.read())

--- PRE-TABLE PARAGRAPHS ---
Introduction to Natural Language Processing
Natural language processing enables computers to understand human language. It is a subfield of artificial intelligence and linguistics. Applications include machine translation, sentiment analysis, and question answering systems.
Modern NLP relies heavily on deep learning architectures. Recurrent neural networks were popular before transformers arrived. Today, transformer-based models dominate the field and achieve state of the art results.
Core Techniques
Tokenization breaks text into individual units. These units may be words, subwords, or characters. Choosing the right granularity affects the performance of downstream models significantly.
   
Common Applications

--- TABLE DATA ---
Application | Description | Difficulty
Translation | Convert between languages | High
Sentiment | Detect emotion in text | Medium
Summarization | Shorten long documents | High



## Task 6: Multi-Document Comparison, Deduplication, and Frequency Report

#### (a) Compare headings between Document A and B with set operations

In [28]:
docA_obj = Document(DOCX_A)
docB_obj = Document(DOCX_B)

headings_A = {p.text for p in docA_obj.paragraphs if p.style.name.startswith("Heading")}
headings_B = {p.text for p in docB_obj.paragraphs if p.style.name.startswith("Heading")}

print("(i)   Headings in BOTH documents:")
for h in sorted(headings_A & headings_B): print("   ", h)
print("(ii)  Headings ONLY in Document A:")
for h in sorted(headings_A - headings_B): print("   ", h)
print("(iii) Headings ONLY in Document B:")
for h in sorted(headings_B - headings_A): print("   ", h)

(i)   Headings in BOTH documents:
    Introduction to Natural Language Processing
(ii)  Headings ONLY in Document A:
    Common Applications
    Core Techniques
(iii) Headings ONLY in Document B:
    Evaluation Metrics


#### (b) Merge Normal paragraphs and deduplicate

In [29]:
normals_A = [p.text for p in docA_obj.paragraphs if p.style.name == "Normal"]
normals_B = [p.text for p in docB_obj.paragraphs if p.style.name == "Normal"]
merged = normals_A + normals_B

seen = set()
deduped = []
for p in merged:
    key = p.strip().lower()
    if key not in seen:
        seen.add(key)
        deduped.append(p)

print(f"Total Normal paragraphs BEFORE deduplication: {len(merged)}")
print(f"Total Normal paragraphs AFTER  deduplication: {len(deduped)}")

Total Normal paragraphs BEFORE deduplication: 8
Total Normal paragraphs AFTER  deduplication: 7


#### (c) Word frequency with `collections.Counter`, top 20

In [30]:
from collections import Counter

counter = Counter()
for p in deduped:
    words = re.sub(r"[^\w\s]", " ", p.lower()).split()
    counter.update(words)

# Remove at least 15 stopwords
STOP_15 = {"the", "is", "a", "an", "and", "or", "of", "to", "in", "on",
           "for", "are", "be", "it", "this"}
for sw in STOP_15:
    counter.pop(sw, None)

print("Top 20 most frequent words:")
for word, count in counter.most_common(20):
    print(f"  {word:<20} {count}")

Top 20 most frequent words:
  language             3
  models               3
  human                2
  applications         2
  nlp                  2
  popular              2
  text                 2
  units                2
  standard             2
  tasks                2
  natural              1
  processing           1
  enables              1
  computers            1
  understand           1
  subfield             1
  artificial           1
  intelligence         1
  linguistics          1
  include              1


#### (d) Summary report

In [31]:
total_headings = len(headings_A | headings_B)
vocab = set(counter.keys())
avg_para_len = round(sum(len(p.split()) for p in deduped) / len(deduped), 2)
longest_para = max(deduped, key=lambda p: len(p.split()))

print("========== SUMMARY REPORT ==========")
print(f"(i)   Total unique headings across both docs : {total_headings}")
print(f"(ii)  Unique body paragraphs (deduplicated)  : {len(deduped)}")
print(f"(iii) Vocabulary size (unique word types)    : {len(vocab)}")
print(f"(iv)  Average paragraph length (words)       : {avg_para_len}")
print(f"(v)   Longest paragraph ({len(longest_para.split())} words):")
print(f"      {longest_para}")

========== SUMMARY REPORT ==========
(i)   Total unique headings across both docs : 4
(ii)  Unique body paragraphs (deduplicated)  : 7
(iii) Vocabulary size (unique word types)    : 108
(iv)  Average paragraph length (words)       : 21.71
(v)   Longest paragraph (31 words):
      Accuracy, precision, recall, and F1 score are standard classification metrics. For generation tasks, BLEU and ROUGE scores measure similarity to reference text. Human evaluation remains the gold standard for assessing quality.


## Task 7: Multi-Endpoint API Data Joining and Analysis

> **Note:** Tasks 7–9 require an internet connection (JSONPlaceholder API).

#### (a) Fetch both endpoints with status validation

In [32]:
import requests

def fetch_json_list(url):
    resp = requests.get(url, timeout=15)
    if resp.status_code != 200:
        raise ConnectionError(f"GET {url} returned status {resp.status_code}")
    return resp.json()

users = fetch_json_list(USERS_URL)
posts = fetch_json_list(POSTS_URL)
print(f"Fetched {len(users)} users and {len(posts)} posts (status 200 OK)")

Fetched 10 users and 100 posts (status 200 OK)


#### (b) Join users + posts into a nested dictionary

In [33]:
# Index posts by userId for efficient joining
posts_by_user = {}
for post in posts:
    posts_by_user.setdefault(post["userId"], []).append(post)

joined = {}
for u in users:
    user_posts = posts_by_user.get(u["id"], [])
    joined[u["name"]] = {
        "email": u["email"],
        "post_count": len(user_posts),
        "titles": [p["title"] for p in user_posts],
    }

# Print first 3 users only
for name in list(joined)[:3]:
    print(f"{name} ({joined[name]['email']}) - {joined[name]['post_count']} posts")
    for t in joined[name]["titles"]:
        print(f"    - {t}")
    print()

Leanne Graham (Sincere@april.biz) - 10 posts
    - sunt aut facere repellat provident occaecati excepturi optio reprehenderit
    - qui est esse
    - ea molestias quasi exercitationem repellat qui ipsa sit aut
    - eum et est occaecati
    - nesciunt quas odio
    - dolorem eum magni eos aperiam quia
    - magnam facilis autem
    - dolorem dolore est ipsam
    - nesciunt iure omnis dolorem tempora et accusantium
    - optio molestias id quia eum

Ervin Howell (Shanna@melissa.tv) - 10 posts
    - et ea vero quia laudantium autem
    - in quibusdam tempore odit est dolorem
    - dolorum ut in voluptas mollitia et saepe quo animi
    - voluptatem eligendi optio
    - eveniet quod temporibus
    - sint suscipit perspiciatis velit dolorum rerum ipsa laboriosam odio
    - fugit voluptas sed molestias voluptatem provident
    - voluptate et itaque vero tempora molestiae
    - adipisci placeat illum aut reiciendis qui
    - doloribus ad provident suscipit at

Clementine Bauch (Nathan@yeseni

#### (c) User with the most posts

In [34]:
top_user_name = max(joined, key=lambda n: joined[n]["post_count"])
top_user = joined[top_user_name]
# find the company name for this user
company = next(u["company"]["name"] for u in users if u["name"] == top_user_name)

print(f"(i)   Full name  : {top_user_name}")
print(f"(ii)  Email      : {top_user['email']}")
print(f"(iii) Company    : {company}")
print(f"(iv)  Post titles:")
for i, t in enumerate(top_user["titles"], 1):
    print(f"      {i}. {t}")

(i)   Full name  : Leanne Graham
(ii)  Email      : Sincere@april.biz
(iii) Company    : Romaguera-Crona
(iv)  Post titles:
      1. sunt aut facere repellat provident occaecati excepturi optio reprehenderit
      2. qui est esse
      3. ea molestias quasi exercitationem repellat qui ipsa sit aut
      4. eum et est occaecati
      5. nesciunt quas odio
      6. dolorem eum magni eos aperiam quia
      7. magnam facilis autem
      8. dolorem dolore est ipsam
      9. nesciunt iure omnis dolorem tempora et accusantium
      10. optio molestias id quia eum


#### (d) Filtered users (company.bs contains 'synergies' OR city length > 8)

In [35]:
print(f"{'Name':<25} {'City':<20} Post count")
print("-" * 60)
for u in users:
    bs = u["company"]["bs"].lower()
    city = u["address"]["city"]
    if "synergies" in bs or len(city) > 8:
        n_posts = len(posts_by_user.get(u["id"], []))
        print(f"{u['name']:<25} {city:<20} {n_posts}")

Name                      City                 Post count
------------------------------------------------------------
Leanne Graham             Gwenborough          10
Ervin Howell              Wisokyburgh          10
Clementine Bauch          McKenziehaven        10
Patricia Lebsack          South Elvis          10
Chelsey Dietrich          Roscoeview           10
Mrs. Dennis Schulist      South Christy        10
Kurtis Weissnat           Howemouth            10
Nicholas Runolfsdottir V  Aliyaview            10
Glenna Reichert           Bartholomebury       10
Clementina DuBuque        Lebsackbury          10


## Task 8: To-Do Completion Rate Analysis Across Users

#### (a) Fetch + validate todos data

In [36]:
import sys

resp = requests.get(TODOS_URL, timeout=15)
if resp.status_code != 200:
    print(f"ERROR: HTTP status {resp.status_code}"); sys.exit(1)

todos = resp.json()
if not isinstance(todos, list):
    print("ERROR: response is not a list"); sys.exit(1)
if len(todos) < 1:
    print("ERROR: todo list is empty"); sys.exit(1)

print(f"Total todo records retrieved: {len(todos)}")

Total todo records retrieved: 200


#### (b) Group by userId and verify counts

In [37]:
stats = {}
for t in todos:
    uid = t["userId"]
    stats.setdefault(uid, {"total": 0, "completed": 0, "incomplete": 0})
    stats[uid]["total"] += 1
    if t["completed"]:
        stats[uid]["completed"] += 1
    else:
        stats[uid]["incomplete"] += 1

print("User | total | completed | incomplete | completed+incomplete==total")
for uid in sorted(stats):
    s = stats[uid]
    ok = (s["completed"] + s["incomplete"] == s["total"])
    print(f"{uid:>4} | {s['total']:>5} | {s['completed']:>9} | "
          f"{s['incomplete']:>10} | {ok}")

User | total | completed | incomplete | completed+incomplete==total
   1 |    20 |        11 |          9 | True
   2 |    20 |         8 |         12 | True
   3 |    20 |         7 |         13 | True
   4 |    20 |         6 |         14 | True
   5 |    20 |        12 |          8 | True
   6 |    20 |         6 |         14 | True
   7 |    20 |         9 |         11 | True
   8 |    20 |        11 |          9 | True
   9 |    20 |         8 |         12 | True
  10 |    20 |        12 |          8 | True


#### (c) Completion rates — highest, lowest, overall average

In [38]:
rates = {uid: round(s["completed"] / s["total"] * 100, 1)
         for uid, s in stats.items()}

highest_uid = max(rates, key=rates.get)
lowest_uid  = min(rates, key=rates.get)
overall_avg = round(sum(rates.values()) / len(rates), 1)

print(f"(i)   Highest completion rate: User {highest_uid} -> {rates[highest_uid]}%")
print(f"(ii)  Lowest  completion rate: User {lowest_uid} -> {rates[lowest_uid]}%")
print(f"(iii) Overall average completion rate: {overall_avg}%")

(i)   Highest completion rate: User 5 -> 60.0%
(ii)  Lowest  completion rate: User 4 -> 30.0%
(iii) Overall average completion rate: 45.0%


#### (d) Formatted leaderboard table (sorted by completion rate)

In [40]:
user_lookup = {u["id"]: u["name"] for u in fetch_json_list(USERS_URL)}

leaderboard = sorted(rates.items(), key=lambda kv: kv[1], reverse=True)

print(f"{'Rank':<6}{'Name':<28}{'Total Tasks':<14}{'Completed':<12}{'Completion Rate %'}")
print("-" * 75)
for rank, (uid, rate) in enumerate(leaderboard, 1):
    print(f"{rank:<6}{user_lookup[uid]:<28}{stats[uid]['total']:<14}"
          f"{stats[uid]['completed']:<12}{rate}")

Rank  Name                        Total Tasks   Completed   Completion Rate %
---------------------------------------------------------------------------
1     Chelsey Dietrich            20            12          60.0
2     Clementina DuBuque          20            12          60.0
3     Leanne Graham               20            11          55.0
4     Nicholas Runolfsdottir V    20            11          55.0
5     Kurtis Weissnat             20            9           45.0
6     Ervin Howell                20            8           40.0
7     Glenna Reichert             20            8           40.0
8     Clementine Bauch            20            7           35.0
9     Patricia Lebsack            20            6           30.0
10    Mrs. Dennis Schulist        20            6           30.0


## Task 9: Nested JSON Flattening, DataFrame Operations, and Comment Analysis

#### (a) `flatten_user()` function applied to all 10 users

In [41]:
def flatten_user(user: dict) -> dict:
    """Flatten a nested JSONPlaceholder user into a single-level dict."""
    return {
        "id": user["id"],
        "name": user["name"],
        "username": user["username"],
        "email": user["email"],
        "city": user["address"]["city"],
        "zipcode": user["address"]["zipcode"],
        "lat": user["address"]["geo"]["lat"],
        "lng": user["address"]["geo"]["lng"],
        "company_name": user["company"]["name"],
        "company_catchphrase": user["company"]["catchPhrase"],
    }

users = fetch_json_list(USERS_URL)          # re-fetch if Task 7 was skipped
flat_users = [flatten_user(u) for u in users]
for fu in flat_users:
    print(fu)

{'id': 1, 'name': 'Leanne Graham', 'username': 'Bret', 'email': 'Sincere@april.biz', 'city': 'Gwenborough', 'zipcode': '92998-3874', 'lat': '-37.3159', 'lng': '81.1496', 'company_name': 'Romaguera-Crona', 'company_catchphrase': 'Multi-layered client-server neural-net'}
{'id': 2, 'name': 'Ervin Howell', 'username': 'Antonette', 'email': 'Shanna@melissa.tv', 'city': 'Wisokyburgh', 'zipcode': '90566-7771', 'lat': '-43.9509', 'lng': '-34.4618', 'company_name': 'Deckow-Crist', 'company_catchphrase': 'Proactive didactic contingency'}
{'id': 3, 'name': 'Clementine Bauch', 'username': 'Samantha', 'email': 'Nathan@yesenia.net', 'city': 'McKenziehaven', 'zipcode': '59590-4157', 'lat': '-68.6102', 'lng': '-47.0653', 'company_name': 'Romaguera-Jacobson', 'company_catchphrase': 'Face to face bifurcated interface'}
{'id': 4, 'name': 'Patricia Lebsack', 'username': 'Karianne', 'email': 'Julianne.OConner@kory.org', 'city': 'South Elvis', 'zipcode': '53919-4257', 'lat': '29.4572', 'lng': '-164.2990', '

#### (b) DataFrame, distance_from_origin, farthest user

In [42]:
import math

df_users = pd.DataFrame(flat_users)
df_users["lat"] = df_users["lat"].astype(float)
df_users["lng"] = df_users["lng"].astype(float)
df_users["distance_from_origin"] = df_users.apply(
    lambda r: math.sqrt(r["lat"] ** 2 + r["lng"] ** 2), axis=1)

print(df_users.to_string(index=False))

farthest = df_users.loc[df_users["distance_from_origin"].idxmax()]
print(f"\nGreatest distance from origin: {farthest['name']} "
      f"({farthest['distance_from_origin']:.4f})")

 id                     name         username                     email           city    zipcode      lat       lng       company_name                      company_catchphrase  distance_from_origin
  1            Leanne Graham             Bret         Sincere@april.biz    Gwenborough 92998-3874 -37.3159   81.1496    Romaguera-Crona   Multi-layered client-server neural-net             89.318161
  2             Ervin Howell        Antonette         Shanna@melissa.tv    Wisokyburgh 90566-7771 -43.9509  -34.4618       Deckow-Crist           Proactive didactic contingency             55.850669
  3         Clementine Bauch         Samantha        Nathan@yesenia.net  McKenziehaven 59590-4157 -68.6102  -47.0653 Romaguera-Jacobson        Face to face bifurcated interface             83.201575
  4         Patricia Lebsack         Karianne Julianne.OConner@kory.org    South Elvis 53919-4257  29.4572 -164.2990      Robel-Corkery Multi-tiered zero tolerance productivity            166.918807
  5  

#### (c) Hemisphere classification

In [43]:
northern = df_users[df_users["lat"] > 0]["name"].tolist()
southern = df_users[df_users["lat"] < 0]["name"].tolist()

print("Northern Hemisphere users:", northern)
print("Southern Hemisphere users:", southern)

avg_north = df_users[df_users["lat"] > 0]["distance_from_origin"].mean()
avg_south = df_users[df_users["lat"] < 0]["distance_from_origin"].mean()
print(f"\nAvg distance - North: {avg_north:.4f}, South: {avg_south:.4f}")
print("Higher average distance_from_origin:",
      "Northern" if avg_north > avg_south else "Southern")

Northern Hemisphere users: ['Patricia Lebsack', 'Kurtis Weissnat', 'Glenna Reichert']
Southern Hemisphere users: ['Leanne Graham', 'Ervin Howell', 'Clementine Bauch', 'Chelsey Dietrich', 'Mrs. Dennis Schulist', 'Nicholas Runolfsdottir V', 'Clementina DuBuque']

Avg distance - North: 123.5833, South: 84.3162
Higher average distance_from_origin: Northern


#### (d) Most-commented and least-commented posts

In [44]:
posts    = fetch_json_list(POSTS_URL)
comments = fetch_json_list(COMMENTS_URL)

comment_counts = {}
for c in comments:
    comment_counts[c["postId"]] = comment_counts.get(c["postId"], 0) + 1

result = [(p["id"], p["title"][:40], comment_counts.get(p["id"], 0)) for p in posts]
result.sort(key=lambda x: x[2], reverse=True)

print("TOP 5 most-commented posts:")
for pid, title, cnt in result[:5]:
    print(f"  #{pid:<4} {cnt:>3} comments | {title}")

print("\nBOTTOM 5 least-commented posts:")
for pid, title, cnt in result[-5:]:
    print(f"  #{pid:<4} {cnt:>3} comments | {title}")

TOP 5 most-commented posts:
  #1      5 comments | sunt aut facere repellat provident occae
  #2      5 comments | qui est esse
  #3      5 comments | ea molestias quasi exercitationem repell
  #4      5 comments | eum et est occaecati
  #5      5 comments | nesciunt quas odio

BOTTOM 5 least-commented posts:
  #96     5 comments | quaerat velit veniam amet cupiditate aut
  #97     5 comments | quas fugiat ut perspiciatis vero provide
  #98     5 comments | laboriosam dolor voluptates
  #99     5 comments | temporibus sit alias delectus eligendi p
  #100    5 comments | at nam consequatur ea labore ea harum
